<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

Система учёта фактур (Invoice Management System)

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>


Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.
    
Требования к базовому классу Invoice:

Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая
сумма (TotalAmount).

Методы:

CalculateTotal(): метод для расчета общей суммы по фактуре.

AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.


Требования к производным классам:
1. ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные
атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен
быть переопределен для добавления информации о дате поставки товара
при добавлении позиции.
2. УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные
атрибуты, такие как Дата оказания услуги (ServiceDate).
Метод RemoveLine() должен быть переопределен для добавления
информации о причине аннулирования услуги при удалении позиции.
3. КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс):
Должна содержать дополнительные атрибуты, такие как Наличие возврата
(ReturnAllowed). Метод CalculateTotal() должен быть переопределен для
учета возможного возврата товара или услуги при расчете общей суммы.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;


public class LineItem
{
    private string _name;
    private decimal _price;
    private int _quantity;

    public string Name
    {
        get => _name;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Наименование товара/услуги не может быть пустым!");
            _name = value;
        }
    }

    public decimal Price
    {
        get => _price;
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Цена не может быть отрицательной!");
            _price = value;
        }
    }

    public int Quantity
    {
        get => _quantity;
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("Количество должно быть больше нуля!");
            _quantity = value;
        }
    }

    public LineItem(string name, decimal price, int quantity)
    {
        Name = name;
        Price = price;
        Quantity = quantity;
    }

    public decimal GetAmount() => Price * Quantity;

    public override string ToString() => $"{Name} — {Quantity} x {Price:0.00} = {GetAmount():0.00}";
}

public class Invoice
{
    private string _invoiceNumber;
    private decimal _totalAmount;

    public string InvoiceNumber
    {
        get => _invoiceNumber;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Номер фактуры не может быть пустым!");
            _invoiceNumber = value;
        }
    }

    public DateTime IssueDate { get; set; }

    public decimal TotalAmount
    {
        get => _totalAmount;
        protected set
        {
            if (value < 0)
                _totalAmount = 0;
            else
                _totalAmount = value;
        }
    }

    public List<LineItem> Lines { get; } = new List<LineItem>();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    public virtual decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (var line in Lines)
        {
            total += line.GetAmount();
        }
        TotalAmount = total;
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) throw new ArgumentNullException(nameof(lineItem));
        Lines.Add(lineItem);
        Console.WriteLine($"Добавлена позиция: {lineItem}");
        CalculateTotal();
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (Lines.Remove(lineItem))
        {
            Console.WriteLine($"Удалена позиция: {lineItem}");
            CalculateTotal();
        }
    }

    public void MergeLinesFrom(Invoice otherInvoice)
    {
        if (otherInvoice == null) return;
        foreach (var line in otherInvoice.Lines)
        {
            AddLine(line);
        }
        Console.WriteLine($"Позиции из фактуры №{otherInvoice.InvoiceNumber} успешно объединены с фактурой №{InvoiceNumber}");
    }

    public void PrintInfo()
    {
        Console.WriteLine($"Фактура №{InvoiceNumber} от {IssueDate:d}, Итоговая сумма: {TotalAmount:0.00}");
    }
}

public class GoodsInvoice : Invoice
{
    private DateTime _supplyDate;

    public DateTime SupplyDate
    {
        get => _supplyDate;
        set
        {
            if (value < IssueDate.Date)
                throw new ArgumentException("Дата поставки не может быть раньше даты выдачи фактуры!");
            _supplyDate = value;
        }
    }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
        : base(invoiceNumber, issueDate)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        Lines.Add(lineItem);
        Console.WriteLine($"[Товар] Добавлена позиция: {lineItem} (Дата поставки: {SupplyDate:d})");
        CalculateTotal();
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate)
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
    }

    public void RemoveLine(LineItem lineItem, string cancellationReason)
    {
        if (Lines.Remove(lineItem))
        {
            Console.WriteLine($"[Услуга аннулирована]: {lineItem}. Причина: {cancellationReason}");
            CalculateTotal();
        }
    }

    public override void RemoveLine(LineItem lineItem)
    {
        RemoveLine(lineItem, "Причина не указана");
    }
}

public class CombinedInvoice : Invoice
{
    private decimal _returnAmount;

    public bool ReturnAllowed { get; set; }

    public decimal ReturnAmount
    {
        get => _returnAmount;
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Сумма возврата не может быть отрицательной!");
            _returnAmount = value;
            CalculateTotal();
        }
    }

    public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed)
        : base(invoiceNumber, issueDate)
    {
        ReturnAllowed = returnAllowed;
    }

    public override decimal CalculateTotal()
    {
        decimal total = base.CalculateTotal();

        if (ReturnAllowed && ReturnAmount > 0)
        {
            total -= ReturnAmount;
            Console.WriteLine($"Учтён возврат на сумму: {ReturnAmount:0.00}");
        }

        TotalAmount = total;
        return TotalAmount;
    }
}

public class Customer
{
    public string Name { get; set; }
    public string Email { get; set; }

    public Customer(string name, string email)
    {
        Name = name;
        Email = email;
    }

    public void PayInvoice(Invoice invoice)
    {
        Console.WriteLine($"Клиент {Name} оплатил фактуру №{invoice.InvoiceNumber} на сумму {invoice.TotalAmount:0.00} руб.");
    }

    public void ApplyDiscount(Invoice invoice, decimal discountPercent)
    {
        decimal discountAmount = invoice.TotalAmount * (discountPercent / 100m);
        Console.WriteLine($"Для клиента {Name} применена скидка {discountPercent}% (-{discountAmount:0.00} руб.) к фактуре №{invoice.InvoiceNumber}");
    }
}

public class InvoiceRegistry
{
    private List<Invoice> _invoices = new List<Invoice>();

    public void AddInvoice(Invoice invoice)
    {
        _invoices.Add(invoice);
    }

    public void DisplaySummary()
    {
        decimal totalSum = 0;
        Console.WriteLine("--- Реестр фактур ---");
        foreach (var inv in _invoices)
        {
            inv.PrintInfo();
            totalSum += inv.TotalAmount;
        }
        Console.WriteLine($"Общая сумма всех фактур в реестре: {totalSum:0.00}");
    }
}


Console.WriteLine("=== 1. Создание объектов через конструкторы с валидацией ===");
Customer customer = new Customer("ООО ТехноПром", "client@technoprom.ru");

GoodsInvoice goodsInvoice = new GoodsInvoice("G-001", DateTime.Now, DateTime.Now.AddDays(3));
goodsInvoice.AddLine(new LineItem("Ноутбук", 55000m, 1));
goodsInvoice.AddLine(new LineItem("Мышь", 1200m, 2));

ServiceInvoice serviceInvoice = new ServiceInvoice("S-001", DateTime.Now, DateTime.Now);
LineItem consulting = new LineItem("Консультация", 3000m, 1);
serviceInvoice.AddLine(consulting);
serviceInvoice.AddLine(new LineItem("Настройка ПО", 5000m, 1));

CombinedInvoice combinedInvoice = new CombinedInvoice("C-001", DateTime.Now, true);
combinedInvoice.AddLine(new LineItem("Принтер", 12000m, 1));
combinedInvoice.AddLine(new LineItem("Обслуживание принтера", 2000m, 1));
combinedInvoice.ReturnAmount = 2000m; // Работа сеттера с проверкой

Console.WriteLine("\n=== 2. Взаимодействие объектов: Покупатель и Фактуры ===");
customer.PayInvoice(goodsInvoice);
customer.ApplyDiscount(serviceInvoice, 10); // Взаимодействие: применение скидки

Console.WriteLine("\n=== 3. Взаимодействие фактур в реестре (InvoiceRegistry) ===");
InvoiceRegistry registry = new InvoiceRegistry();
registry.AddInvoice(goodsInvoice);
registry.AddInvoice(serviceInvoice);
registry.AddInvoice(combinedInvoice);

registry.DisplaySummary();

Console.WriteLine("\n=== 4. Объединение позиций одной фактуры с другой ===");
GoodsInvoice extraGoods = new GoodsInvoice("G-002", DateTime.Now, DateTime.Now.AddDays(1));
extraGoods.AddLine(new LineItem("Клавиатура", 3500m, 1));

goodsInvoice.MergeLinesFrom(extraGoods); // Взаимодействие объектов фактур напрямую
goodsInvoice.PrintInfo();


The below script needs to be able to find the current output cell; this is an easy method to get it.

=== 1. Создание объектов через конструкторы с валидацией ===
[Товар] Добавлена позиция: Ноутбук — 1 x 55000,00 = 55000,00 (Дата поставки: 27.09.2026)
[Товар] Добавлена позиция: Мышь — 2 x 1200,00 = 2400,00 (Дата поставки: 27.09.2026)
Добавлена позиция: Консультация — 1 x 3000,00 = 3000,00
Добавлена позиция: Настройка ПО — 1 x 5000,00 = 5000,00
Добавлена позиция: Принтер — 1 x 12000,00 = 12000,00
Добавлена позиция: Обслуживание принтера — 1 x 2000,00 = 2000,00
Учтён возврат на сумму: 2000,00

=== 2. Взаимодействие объектов: Покупатель и Фактуры ===
Клиент ООО ТехноПром оплатил фактуру №G-001 на сумму 57400,00 руб.
Для клиента ООО ТехноПром применена скидка 10% (-800,00 руб.) к фактуре №S-001

=== 3. Взаимодействие фактур в реестре (InvoiceRegistry) ===
--- Реестр фактур ---
Фактура №G-001 от 24.09.2026, Итоговая сумма: 57400,00
Фактура №S-001 от 24.09.2026, Итоговая сумма: 8000,00
Фактура №C-001 от 24.09.2026, Итоговая сумма: 12000,00
Общая сумма всех фактур в реестре: 77400,00

=== 4. 